<a href="https://colab.research.google.com/github/Andre-Sil/Andre-Sil/blob/main/b3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================
# 📈 GEBRA Portfolio v16.0 – Confluência Total com Anatomia Completa
# ================================

import os

# ==================== CONFIGURAÇÕES DE ACESSO ====================
EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')
BRAPI_TOKEN = os.getenv('BRAPI_API_TOKEN', '') or os.getenv('BRAPI_TOKEN', '')
TELEGRAM_TOKEN = os.getenv('TELEGRAM_TOKEN', '')
TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID', '')

# ==================== PARÂMETROS GLOBAIS ====================
CAPITAL_TOTAL = 100000.0
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.10
PRECO_MINIMO = 4.00
MAX_SETUPS_POR_DIA = 5
MAX_ATIVOS_POR_SETOR = 2
PAYOFF_MINIMO = 3.0

# --- Parâmetros de padrões ---
TOLERANCIA_NIVEL = 0.02
TOLERANCIA_OMBRO = 0.05
CORPO_MINIMO_CANDLE = 0.70
FECHAMENTO_EXTREMIDADE = 0.10
VOLUME_MULT_ALTO = 2.0
VOLUME_MULT_MEDIO = 1.5
VOLUME_MULT_PIVO = 1.3
ATR_PERIODOS = 14
VOLUME_FORMACAO_MAX_MEDIA = 0.5
CUP_TOPO_CORRECAO_MAX = 0.50
CUP_HANDLE_MAX_DIAS = 14

# --- Confluência ---
PONTUACAO_MINIMA_CONFLUENCIA = 75
MIN_SINAIS_CONFLUENCIA = 3
IFR_MAX_COMPRA = 70
IFR_MIN_COMPRA = 25
IFR_MAX_VENDA = 75
IFR_MIN_VENDA = 30

# --- Filtros de liquidez e dados ---
VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
ADX_MINIMO = 25
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
FALLBACK_TICKERS = ['PETR4','VALE3','ITUB4','BBDC4','BBAS3','ABEV3','WEGE3','RADL3','SUZB3','GGBR4','MGLU3','VVAR3','RENT3','RAIL3','CCRO3','ELET3','CPFE3','SBSP3','SANB11','B3SA3','JBSS3','BRFS3','KLBN11','EQTL3']
TICKERS_BLOQUEADOS = ['GFSA3.SA','ONCO3.SA','PMAM3.SA','AZTE3.SA','RAIZ4.SA','BHIA3.SA','CASH3.SA','LJQQ3.SA','RCSL4.SA','HBOR3.SA']

# --- Timeframes ---
ANALISAR_DIARIO = True
ANALISAR_SEMANAL = True
ANALISAR_MENSAL = True

# --- Modo teste ---
MODO_TESTE = False
TESTE_TICKERS = ['PETR4','VALE3','ITUB4','BBDC4','BBAS3','ABEV3','WEGE3','RADL3','GGBR4','MGLU3']

# --- Logging ---
ARQUIVO_LOG = "trading_log_v16.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v16.log"

# --- Novas feature flags ---
MODO_PULLBACK = True           # Permitir entrada via pullback
USAR_OBV = True                # Checar OBV
USAR_DIVERGENCIAS = True       # Checar divergências IFR/MACD
USAR_GAPS = True               # Detectar gaps
USAR_BOLLINGER = True          # Bollinger Bands squeeze
USAR_MACD_CONFLUENCIA = True   # MACD como sinal de confluência

# --- Novos thresholds ---
GAP_MIN_ATR_MULT = 0.5         # Gap mínimo em múltiplos do ATR
GAP_VOLUME_MULT = 1.5          # Volume mínimo para gap de fuga
GAP_EXAUSTAO_VOLUME_MULT = 2.0 # Volume para gap de exaustão
BOLLINGER_SQUEEZE_LOOKBACK = 120
BOLLINGER_SQUEEZE_TOL = 1.1
PULLBACK_TOLERANCIA_PCT = 0.005
PULLBACK_VOLUME_MAX_PCT = 0.8

print("✅ Parâmetros v16.0 carregados")

✅ Parâmetros v16.0 carregados


In [2]:
# ================ INSTALAÇÃO E IMPORTAÇÕES ================
!pip install yfinance pandas-ta --quiet 2>/dev/null

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, sys, traceback, gc, socket
from collections import Counter

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

try:
    from scipy.signal import argrelextrema
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("⚠️ scipy não disponível. Usando detecção manual de pivôs.")

print("✅ Bibliotecas carregadas")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 10.2 MB/s eta 0:00:00
⚠️ scipy não disponível. Usando detecção manual de pivôs.
✅ Bibliotecas carregadas


In [3]:
# ================ LOGGER E UTILITÁRIOS ================
class Logger:
    def __init__(self, log_json, log_detalhado):
        self.log_json = log_json
        self.log_detalhado = log_detalhado
        self.t0 = time.time()
        self.tm = {}
        self.buffer = []
        self.max_buffer = 100

    def log(self, msg, nivel="INFO", extra=None):
        ts = datetime.now().strftime("%H:%M:%S")
        linha = f"[{ts}] [{nivel}] {msg}"
        if extra:
            linha += f" | {extra}"
        print(linha)
        if self.log_detalhado:
            self.buffer.append(linha + "\n")
            if len(self.buffer) >= self.max_buffer:
                self._flush()

    def _flush(self):
        if self.log_detalhado and self.buffer:
            try:
                with open(self.log_detalhado, 'a', encoding='utf-8') as f:
                    f.writelines(self.buffer)
                self.buffer.clear()
            except Exception as e:
                print(f"Erro ao gravar log: {e}")

    def warn(self, msg, extra=None):
        self.log(msg, "WARN", extra)

    def error(self, msg, extra=None):
        self.log(msg, "ERRO", extra)

    def inicio(self, etapa):
        self.tm[etapa] = {'ini': time.time()}
        self.log(f"🚀 INÍCIO: {etapa}", "ETAPA")

    def fim(self, etapa, dados=None):
        if etapa in self.tm:
            dur = time.time() - self.tm[etapa]['ini']
            self.tm[etapa]['dur'] = dur
            msg = f"✅ FIM: {etapa} ({dur:.1f}s)"
            if dados:
                msg += " | " + " | ".join(f"{k}:{v}" for k, v in dados.items())
            self.log(msg, "ETAPA")

    def resumo(self):
        self._flush()
        total = time.time() - self.t0
        self.log("\n" + "="*60, "RESUMO")
        self.log(f"⏱️ Tempo total: {total:.1f}s", "RESUMO")
        for etapa, dados in self.tm.items():
            if 'dur' in dados:
                pct = dados['dur']/total*100 if total>0 else 0
                self.log(f"   • {etapa}: {dados['dur']:.1f}s ({pct:.0f}%)", "RESUMO")
        self.log("="*60 + "\n", "RESUMO")

logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)

def _log_exc(contexto, e):
    try:
        msg = f"[{contexto}] {type(e).__name__}: {str(e)[:200]}"
        logger.error(msg)
        with open('traceback_errors.log', 'a', encoding='utf-8') as f:
            f.write(f"\n{'='*60}\n{datetime.now()}\n{contexto}\n{str(e)}\n{traceback.format_exc()}")
    except:
        pass

def enviar_telegram(mensagem, parse_mode='HTML'):
    if not TELEGRAM_TOKEN or not TELEGRAM_CHAT_ID:
        return
    try:
        requests.post(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage",
                      data={'chat_id': TELEGRAM_CHAT_ID, 'text': mensagem, 'parse_mode': parse_mode}, timeout=10)
    except Exception as e:
        _log_exc('Telegram', e)

def verificar_conectividade():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=5)
        return True
    except OSError:
        return False

print("✅ Logger e utilitários carregados")

✅ Logger e utilitários carregados


In [4]:
# ================ DOWNLOAD DE DADOS (BRAPI + YFINANCE) ================
def baixar_dados_brapi(tickers_sa, periodo_anos=5, interval='1d'):
    if not BRAPI_TOKEN:
        return {}
    tickers_limpos = [t.replace('.SA', '') for t in tickers_sa]
    resultados = {}
    erros_http = {}
    for i, ticker in enumerate(tickers_limpos):
        if i % 10 == 0:
            logger.log(f"📡 BRAPI ({interval}): {i+1}/{len(tickers_limpos)}")
        url = f"https://brapi.dev/api/quote/{ticker}"
        params = {'range': f'{periodo_anos}y', 'interval': interval, 'fundamental': 'false', 'token': BRAPI_TOKEN}
        for tentativa in range(2):
            try:
                resp = requests.get(url, params=params, timeout=30)
                if resp.status_code == 200:
                    data = resp.json()
                    quotes = data.get('results', [data])
                    if not isinstance(quotes, list):
                        quotes = [quotes]
                    for quote in quotes:
                        tk = quote['symbol'] + '.SA'
                        if 'historicalDataPrice' in quote:
                            df = pd.DataFrame(quote['historicalDataPrice'])
                            df['date'] = pd.to_datetime(df['date'], unit='s')
                            df.set_index('date', inplace=True)
                            df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'}, inplace=True)
                            df = df[['Open','High','Low','Close','Volume']]
                            resultados[tk] = df
                    break
                elif resp.status_code == 429:
                    time.sleep(2 ** (tentativa + 1))
                else:
                    if tentativa == 1:
                        erros_http[resp.status_code] = erros_http.get(resp.status_code, 0) + 1
                    break
            except Exception as e:
                _log_exc(f'BRAPI {ticker}', e)
                break
        time.sleep(0.25)
    if erros_http:
        logger.warn(f"BRAPI erros HTTP: {erros_http}")
    logger.log(f"📦 BRAPI ({interval}): {len(resultados)} tickers")
    return resultados

def baixar_dados_yfinance_v2(tickers_sa, periodo='5y'):
    data = {}
    for i, t in enumerate(tickers_sa):
        if i % 20 == 0:
            logger.log(f"🔄 yfinance: {i+1}/{len(tickers_sa)}")
        try:
            df = yf.Ticker(t).history(period=periodo, auto_adjust=True)
            if df is not None and not df.empty and 'Close' in df.columns:
                df = df[['Open','High','Low','Close','Volume']].copy()
                data[t] = df
        except Exception as e:
            _log_exc(f'yfinance {t}', e)
        time.sleep(0.3)
    logger.log(f"📦 yfinance: {len(data)} tickers")
    return data

def obter_tickers_brapi():
    if not BRAPI_TOKEN:
        return []
    tickers = []
    try:
        url = "https://brapi.dev/api/quote/list"
        headers = {'Authorization': f'Bearer {BRAPI_TOKEN}'}
        page = 1
        while True:
            params = {'limit': 100, 'page': page, 'type': 'stock'}
            resp = requests.get(url, headers=headers, params=params, timeout=15)
            if resp.status_code != 200:
                break
            data = resp.json()
            stocks = data.get('stocks', [])
            if not stocks:
                break
            for s in stocks:
                tickers.append(s['stock'])
            page += 1
            time.sleep(0.2)
        logger.log(f"📋 BRAPI list: {len(tickers)} tickers")
    except Exception as e:
        logger.warn(f"Erro ao obter lista da brapi: {e}")
    return tickers

def obter_tickers_scraping():
    try:
        resp = requests.get("https://www.dadosdemercado.com.br/acoes", timeout=10, headers={'User-Agent':'Mozilla/5.0'})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')
        tickers = []
        for row in soup.select('table tbody tr'):
            cells = row.find_all('td')
            if cells and not cells[0].text.strip().startswith('#'):
                tickers.append(cells[0].text.strip().replace('.SA', ''))
        return tickers
    except Exception as e:
        logger.warn(f"Scraping falhou: {e}")
        return []

def obter_tickers_b3():
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE) as f:
                cache = json.load(f)
            if (datetime.now() - datetime.fromisoformat(cache['timestamp'])).total_seconds() / 3600 < 24:
                logger.log(f"📦 Cache tickers ({len(cache['tickers'])} ativos)")
                return cache['tickers']
        except:
            pass
    tickers = []
    if BRAPI_TOKEN:
        tickers = obter_tickers_brapi()
    if not tickers:
        tickers = obter_tickers_scraping()
    if not tickers:
        tickers = FALLBACK_TICKERS.copy()
        logger.warn("Usando lista fallback de tickers")
    if tickers:
        with open(CACHE_TICKERS_FILE, 'w') as f:
            json.dump({'timestamp': datetime.now().isoformat(), 'tickers': tickers}, f)
    return tickers

print("✅ Módulo de download carregado")

✅ Módulo de download carregado


In [5]:
# ================ FUNÇÕES AUXILIARES E INDICADORES ================
def _safe_divide(a, b, default=np.nan):
    return default if b is None or b == 0 or pd.isna(b) else a / b

def encontrar_picos(series, ordem=5, modo='max'):
    if SCIPY_AVAILABLE:
        if modo == 'max':
            indices = argrelextrema(series.values, np.greater, order=ordem)[0]
        else:
            indices = argrelextrema(series.values, np.less, order=ordem)[0]
        return [(int(i), series.iloc[i]) for i in indices if i > ordem and i < len(series)-ordem]
    else:
        pivos = []
        for i in range(ordem, len(series)-ordem):
            if modo == 'max' and series.iloc[i] == max(series.iloc[i-ordem:i+ordem+1]):
                pivos.append((i, series.iloc[i]))
            elif modo == 'min' and series.iloc[i] == min(series.iloc[i-ordem:i+ordem+1]):
                pivos.append((i, series.iloc[i]))
        return pivos

def calcular_fibonacci_retracao(df, n_barras=50):
    if len(df) < n_barras:
        return None
    highs = df['High']
    lows = df['Low']
    max_idx = highs.rolling(n_barras).max().idxmax()
    min_idx = lows.rolling(n_barras).min().idxmin()
    if max_idx < min_idx:
        topo = highs.loc[max_idx:min_idx].max()
        fundo = lows.loc[max_idx:min_idx].min()
    else:
        topo = highs.loc[min_idx:max_idx].max()
        fundo = lows.loc[min_idx:max_idx].min()
    if topo <= fundo or fundo <= 0:
        return None
    diff = topo - fundo
    return {'38.2%': round(topo - diff*0.382, 2),
            '50.0%': round(topo - diff*0.5, 2),
            '61.8%': round(topo - diff*0.618, 2)}

def preco_em_zona_interesse(df, preco, direcao, tolerancia=0.02):
    suportes = [v for _, v in encontrar_picos(df['Low'], ordem=5, modo='min')[-10:]]
    resistencias = [v for _, v in encontrar_picos(df['High'], ordem=5, modo='max')[-10:]]
    fib = calcular_fibonacci_retracao(df)
    niveis = []
    if fib:
        niveis.extend([fib['38.2%'], fib['50.0%'], fib['61.8%']])
    if direcao == 'COMPRA':
        for sup in suportes:
            if abs(preco - sup) / sup <= tolerancia:
                return True
        for niv in niveis:
            if abs(preco - niv) / niv <= tolerancia:
                return True
    else:
        for res in resistencias:
            if abs(res - preco) / preco <= tolerancia:
                return True
        for niv in niveis:
            if abs(niv - preco) / preco <= tolerancia:
                return True
    return False

def verificar_medias_suporte(df, direcao):
    if len(df) < 200:
        return False
    mm20 = df['Close'].rolling(20).mean().iloc[-1]
    mm50 = df['Close'].rolling(50).mean().iloc[-1]
    mm200 = df['Close'].rolling(200).mean().iloc[-1]
    preco = df['Close'].iloc[-1]
    if direcao == 'COMPRA':
        return preco > mm20 and mm20 > mm50 and mm50 > mm200
    else:
        return preco < mm20 and mm20 < mm50 and mm50 < mm200

def calcular_atr(df, periodo=ATR_PERIODOS):
    try:
        atr_serie = ta.atr(df['High'], df['Low'], df['Close'], length=periodo)
        if atr_serie is None or atr_serie.empty:
            return 0.0
        return float(atr_serie.iloc[-1])
    except:
        return 0.0

print("✅ Funções auxiliares carregadas")

✅ Funções auxiliares carregadas


In [6]:
# ================ NOVOS DETECTORES: PADRÕES DE VELAS ================
def detectar_martelo(row, direcao='COMPRA'):
    corpo = abs(row['Close'] - row['Open'])
    range_total = row['High'] - row['Low']
    if range_total == 0:
        return False
    if corpo > range_total * 0.35:
        return False
    if direcao == 'COMPRA':
        sombra_inferior = min(row['Open'], row['Close']) - row['Low']
        sombra_superior = row['High'] - max(row['Open'], row['Close'])
        return sombra_inferior >= 2 * corpo and sombra_superior <= corpo * 0.5
    else:
        return False

def detectar_estrela_cadente(row, direcao='VENDA'):
    corpo = abs(row['Close'] - row['Open'])
    range_total = row['High'] - row['Low']
    if range_total == 0:
        return False
    if corpo > range_total * 0.35:
        return False
    if direcao == 'VENDA':
        sombra_superior = row['High'] - max(row['Open'], row['Close'])
        sombra_inferior = min(row['Open'], row['Close']) - row['Low']
        return sombra_superior >= 2 * corpo and sombra_inferior <= corpo * 0.5
    return False

def detectar_doji(row):
    corpo = abs(row['Close'] - row['Open'])
    range_total = row['High'] - row['Low']
    if range_total == 0:
        return True
    return corpo <= range_total * 0.05

def detectar_engolfo(row, row_anterior, direcao='COMPRA'):
    corpo_atual = abs(row['Close'] - row['Open'])
    corpo_ant = abs(row_anterior['Close'] - row_anterior['Open'])
    if corpo_ant == 0:
        return False
    if direcao == 'COMPRA':
        return (row['Close'] > row['Open'] and
                row_anterior['Close'] < row_anterior['Open'] and
                row['Open'] <= row_anterior['Close'] and
                row['Close'] >= row_anterior['Open'])
    else:  # VENDA
        return (row['Close'] < row['Open'] and
                row_anterior['Close'] > row_anterior['Open'] and
                row['Open'] >= row_anterior['Close'] and
                row['Close'] <= row_anterior['Open'])

def validar_candle_forca_v2(row, direcao='COMPRA'):
    """Força básica: corpo >= 70% do range e fechamento no extremo."""
    rng = row['High'] - row['Low']
    if rng <= 0:
        return False, "range zero"
    corpo = abs(row['Close'] - row['Open'])
    if corpo / rng < CORPO_MINIMO_CANDLE:
        return False, f"corpo/range = {corpo/rng:.2f} < {CORPO_MINIMO_CANDLE}"
    if direcao == 'COMPRA':
        fech_rel = (row['Close'] - row['Low']) / rng
        if fech_rel < (1 - FECHAMENTO_EXTREMIDADE):
            return False, f"fechamento a {fech_rel:.0%} da mínima"
    else:
        fech_rel = (row['High'] - row['Close']) / rng
        if fech_rel < (1 - FECHAMENTO_EXTREMIDADE):
            return False, f"fechamento a {fech_rel:.0%} da máxima"
    return True, "OK"

def classificar_candle(row, row_anterior, direcao):
    """Retorna lista de padrões de candle detectados."""
    padroes = []
    if detectar_doji(row):
        padroes.append('Doji')
    if detectar_martelo(row, direcao):
        padroes.append('Martelo')
    if detectar_estrela_cadente(row, direcao):
        padroes.append('Estrela Cadente')
    if row_anterior is not None:
        if detectar_engolfo(row, row_anterior, direcao):
            padroes.append('Engolfo')
    # Corpo longo (Marubozu-like)
    corpo = abs(row['Close'] - row['Open'])
    rng = row['High'] - row['Low']
    if rng > 0 and corpo / rng >= 0.8:
        padroes.append('Corpo Longo')
    return padroes

print("✅ Detectores de velas carregados")

✅ Detectores de velas carregados


In [7]:
# ================ OBV E DIVERGÊNCIAS ================
def calcular_obv(df):
    obv = [0]
    for i in range(1, len(df)):
        if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
            obv.append(obv[-1] + df['Volume'].iloc[i])
        elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
            obv.append(obv[-1] - df['Volume'].iloc[i])
        else:
            obv.append(obv[-1])
    return pd.Series(obv, index=df.index)

def obv_antecipacao(df, direcao):
    """
    Verifica se OBV já rompeu sua resistência/suporte antes do preço.
    Para COMPRA: OBV atual > OBV no último topo de preço.
    Para VENDA: OBV atual < OBV no último fundo de preço.
    """
    if len(df) < 50:
        return False
    obv = calcular_obv(df)
    if direcao == 'COMPRA':
        topos_preco = encontrar_picos(df['High'], ordem=5, modo='max')
        if len(topos_preco) < 1:
            return False
        idx_topo, valor_topo = topos_preco[-1]
        obv_topo = obv.iloc[idx_topo]
        return obv.iloc[-1] > obv_topo and df['Close'].iloc[-1] < valor_topo
    else:
        fundos_preco = encontrar_picos(df['Low'], ordem=5, modo='min')
        if len(fundos_preco) < 1:
            return False
        idx_fundo, valor_fundo = fundos_preco[-1]
        obv_fundo = obv.iloc[idx_fundo]
        return obv.iloc[-1] < obv_fundo and df['Close'].iloc[-1] > valor_fundo

def detectar_divergencia_oscilador(df, oscilador='IFR', direcao='COMPRA'):
    """
    Verifica divergência usando os últimos 2 pivôs.
    """
    if len(df) < 50:
        return False
    if oscilador == 'IFR':
        serie_osc = ta.rsi(df['Close'], length=14)
    elif oscilador == 'MACD':
        macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
        serie_osc = macd['MACD_12_26_9']
    else:
        return False

    if direcao == 'COMPRA':
        fundos_preco = encontrar_picos(df['Low'], modo='min', ordem=5)
        if len(fundos_preco) < 2:
            return False
        f1 = fundos_preco[-2]
        f2 = fundos_preco[-1]
        if f2[1] < f1[1]:
            osc1 = serie_osc.iloc[f1[0]]
            osc2 = serie_osc.iloc[f2[0]]
            if pd.notna(osc1) and pd.notna(osc2) and osc2 > osc1:
                return True
    else:
        topos_preco = encontrar_picos(df['High'], modo='max', ordem=5)
        if len(topos_preco) < 2:
            return False
        t1 = topos_preco[-2]
        t2 = topos_preco[-1]
        if t2[1] > t1[1]:
            osc1 = serie_osc.iloc[t1[0]]
            osc2 = serie_osc.iloc[t2[0]]
            if pd.notna(osc1) and pd.notna(osc2) and osc2 < osc1:
                return True
    return False

print("✅ OBV e Divergências carregados")

✅ OBV e Divergências carregados


In [8]:
# ================ GAPS ================
def detectar_gap(df, i=-1):
    """Retorna (tipo, tamanho_pct) do gap no índice i. None se não houver gap."""
    if len(df) < abs(i)+2:
        return None
    row = df.iloc[i]
    row_ant = df.iloc[i-1]
    if row['Low'] > row_ant['High']:
        gap_size = (row['Low'] - row_ant['High']) / row_ant['High']
        return ('ALTA', gap_size)
    elif row['High'] < row_ant['Low']:
        gap_size = (row_ant['Low'] - row['High']) / row_ant['Low']
        return ('BAIXA', gap_size)
    return None

def classificar_gap(df, atr, media_volume):
    """
    Classifica o gap no último candle.
    Retorna: None, 'Fuga', 'Exaustão', 'Comum'
    """
    gap_info = detectar_gap(df)
    if gap_info is None:
        return None
    tipo, tamanho = gap_info
    if tamanho * df['Close'].iloc[-1] < GAP_MIN_ATR_MULT * atr:
        return None  # Gap insignificante
    vol = df['Volume'].iloc[-1]

    # Gap de Fuga: volume > 1.5x média
    if vol > media_volume * GAP_VOLUME_MULT:
        return 'Fuga'

    # Gap de Exaustão: volume massivo (2x), candle pequeno/sombra longa, tendência esticada?
    row = df.iloc[-1]
    corpo = abs(row['Close'] - row['Open'])
    rng = row['High'] - row['Low']
    corpo_pequeno = (rng > 0) and (corpo / rng < 0.5)
    if vol > media_volume * GAP_EXAUSTAO_VOLUME_MULT and corpo_pequeno:
        # Verificar se está esticado (acima da MME20 por vários dias)
        mme20 = df['Close'].ewm(span=20).mean()
        acima_mme = (df['Close'] > mme20).tail(15).sum()
        if acima_mme >= 15:
            return 'Exaustão'
    return 'Comum'

print("✅ Detector de Gaps carregado")

✅ Detector de Gaps carregado


In [9]:
# ================ FIBONACCI ALVOS ================
def calcular_expansao_fibonacci(df, direcao):
    """
    Calcula projeções 161.8% e 261.8% a partir do último pivô (topo/fundo).
    Retorna dict com alvos.
    """
    if len(df) < 30:
        return None
    if direcao == 'COMPRA':
        fundos = encontrar_picos(df['Low'], ordem=5, modo='min')
        topos = encontrar_picos(df['High'], ordem=5, modo='max')
        if len(fundos) < 2 or len(topos) < 1:
            return None
        # Pivô de alta simplificado: último fundo -> topo -> fundo
        fundo1 = fundos[-2][1]
        topo = max([t[1] for t in topos if t[0] > fundos[-2][0]] + [fundos[-1][1]])
        fundo2 = fundos[-1][1]
        diff = topo - fundo1
        if diff <= 0:
            return None
        return {
            '161.8%': round(fundo2 + diff * 1.618, 2),
            '261.8%': round(fundo2 + diff * 2.618, 2)
        }
    else:
        topos = encontrar_picos(df['High'], ordem=5, modo='max')
        fundos = encontrar_picos(df['Low'], ordem=5, modo='min')
        if len(topos) < 2 or len(fundos) < 1:
            return None
        topo1 = topos[-2][1]
        fundo = min([f[1] for f in fundos if f[0] > topos[-2][0]] + [topos[-1][1]])
        topo2 = topos[-1][1]
        diff = topo1 - fundo
        if diff <= 0:
            return None
        return {
            '161.8%': round(topo2 - diff * 1.618, 2),
            '261.8%': round(topo2 - diff * 2.618, 2)
        }

print("✅ Fibonacci alvos carregado")

✅ Fibonacci alvos carregado


In [10]:
# ================ PULLBACK ================
def verificar_pullback(df, linha_rompida, direcao, media_volume):
    """
    Verifica se houve pullback nos últimos candles.
    direcao: 'COMPRA' (rompeu resistência para cima, recua para a linha como suporte)
    """
    if len(df) < 3:
        return False, None
    # Buscar nos últimos 5 candles toque na linha
    for i in range(-5, 0):
        row = df.iloc[i]
        if direcao == 'COMPRA':
            if row['Low'] <= linha_rompida * (1 + PULLBACK_TOLERANCIA_PCT) and row['Low'] >= linha_rompida * (1 - PULLBACK_TOLERANCIA_PCT):
                # Verificar volume baixo
                if df['Volume'].iloc[i] < media_volume * PULLBACK_VOLUME_MAX_PCT:
                    # Verificar candle de confirmação (precisa ser Martelo/Engolfo)
                    row_ant = df.iloc[i-1] if abs(i) < len(df) else None
                    if detectar_martelo(row, 'COMPRA') or (row_ant is not None and detectar_engolfo(row, row_ant, 'COMPRA')):
                        return True, row
        else:  # VENDA, repique
            if row['High'] <= linha_rompida * (1 + PULLBACK_TOLERANCIA_PCT) and row['High'] >= linha_rompida * (1 - PULLBACK_TOLERANCIA_PCT):
                if df['Volume'].iloc[i] < media_volume * PULLBACK_VOLUME_MAX_PCT:
                    row_ant = df.iloc[i-1] if abs(i) < len(df) else None
                    if detectar_estrela_cadente(row, 'VENDA') or (row_ant is not None and detectar_engolfo(row, row_ant, 'VENDA')):
                        return True, row
    return False, None

print("✅ Detector de Pullback carregado")

✅ Detector de Pullback carregado


In [11]:
# ================ BOLLINGER BANDS ================
def calcular_bollinger_squeeze(df, lookback=BOLLINGER_SQUEEZE_LOOKBACK, tol=BOLLINGER_SQUEEZE_TOL):
    if len(df) < lookback:
        return False
    bbands = ta.bbands(df['Close'], length=20, std=2)
    if bbands is None:
        return False
    banda_sup = bbands['BBU_20_2.0']
    banda_inf = bbands['BBL_20_2.0']
    banda_media = bbands['BBM_20_2.0']
    largura_atual = (banda_sup.iloc[-1] - banda_inf.iloc[-1]) / banda_media.iloc[-1]
    historico_largura = (banda_sup - banda_inf) / banda_media
    minimo_historico = historico_largura.rolling(lookback).min().iloc[-1]
    return largura_atual <= minimo_historico * tol

print("✅ Bollinger Bands carregado")

✅ Bollinger Bands carregado


In [12]:
# ================ DETECÇÃO DE PADRÕES GRÁFICOS ================
def _mesmo_nivel(vals, tol=TOLERANCIA_NIVEL):
    if len(vals) < 2: return True
    ref = vals[0]
    return all(abs(v - ref) / ref <= tol for v in vals)

def detectar_triangulo_simetrico(df, min_pontos=2):
    if len(df) < 30: return False, {}
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_pontos*2:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_pontos*2:]
    if len(highs) < min_pontos or len(lows) < min_pontos: return False, {}
    topos = [v for _, v in highs[-min_pontos:]]
    fundos = [v for _, v in lows[-min_pontos:]]
    topos_desc = all(topos[i] > topos[i+1] for i in range(len(topos)-1))
    fundos_asc = all(fundos[i] < fundos[i+1] for i in range(len(fundos)-1))
    if topos_desc and fundos_asc:
        altura = max(topos) - min(fundos)
        return True, {'tipo': 'Triângulo Simétrico', 'altura': altura, 'topos': topos, 'fundos': fundos}
    return False, {}

def detectar_triangulo_ascendente(df, min_pontos=2):
    if len(df) < 30: return False, {}
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_pontos*2:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_pontos*2:]
    if len(highs) < min_pontos or len(lows) < min_pontos: return False, {}
    topos = [v for _, v in highs[-min_pontos:]]
    fundos = [v for _, v in lows[-min_pontos:]]
    if _mesmo_nivel(topos) and all(fundos[i] < fundos[i+1] for i in range(len(fundos)-1)):
        altura = max(topos) - min(fundos)
        return True, {'tipo': 'Triângulo Ascendente', 'altura': altura, 'resistencia': topos[0]}
    return False, {}

def detectar_triangulo_descendente(df, min_pontos=2):
    if len(df) < 30: return False, {}
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_pontos*2:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_pontos*2:]
    if len(highs) < min_pontos or len(lows) < min_pontos: return False, {}
    topos = [v for _, v in highs]
    fundos = [v for _, v in lows]
    if _mesmo_nivel(fundos) and all(topos[i] > topos[i+1] for i in range(len(topos)-1)):
        altura = max(topos) - min(fundos)
        return True, {'tipo': 'Triângulo Descendente', 'altura': altura, 'suporte': fundos[0]}
    return False, {}

def detectar_retangulo(df, min_toques=2):
    if len(df) < 20: return False, {}
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_toques:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_toques:]
    if len(highs) < min_toques or len(lows) < min_toques: return False, {}
    topos = [v for _, v in highs]
    fundos = [v for _, v in lows]
    if _mesmo_nivel(topos) and _mesmo_nivel(fundos):
        altura = topos[0] - fundos[0]
        return True, {'tipo': 'Retângulo', 'altura': altura, 'resistencia': topos[0], 'suporte': fundos[0]}
    return False, {}

def detectar_oco(df, tipo='topo'):
    if len(df) < 50: return False, {}
    if tipo == 'topo':
        picos = encontrar_picos(df['High'], ordem=5, modo='max')
        if len(picos) < 3: return False, {}
        ombro_e, cabeca, ombro_d = picos[-3], picos[-2], picos[-1]
        if not (cabeca[1] > ombro_e[1] and cabeca[1] > ombro_d[1]): return False, {}
        if abs(ombro_e[1] - ombro_d[1]) / ombro_e[1] > TOLERANCIA_OMBRO: return False, {}
        vale1 = min(df['Low'].iloc[ombro_e[0]:cabeca[0]])
        vale2 = min(df['Low'].iloc[cabeca[0]:ombro_d[0]])
        neckline = (vale1 + vale2) / 2
        altura = cabeca[1] - neckline
        return True, {'tipo': 'OCO Topo', 'altura': altura, 'cabeca': cabeca[1], 'ombro_dir': ombro_d[1], 'neckline': neckline}
    else:
        vales = encontrar_picos(df['Low'], ordem=5, modo='min')
        if len(vales) < 3: return False, {}
        ombro_e, cabeca, ombro_d = vales[-3], vales[-2], vales[-1]
        if not (cabeca[1] < ombro_e[1] and cabeca[1] < ombro_d[1]): return False, {}
        if abs(ombro_e[1] - ombro_d[1]) / ombro_e[1] > TOLERANCIA_OMBRO: return False, {}
        pico1 = max(df['High'].iloc[ombro_e[0]:cabeca[0]])
        pico2 = max(df['High'].iloc[cabeca[0]:ombro_d[0]])
        neckline = (pico1 + pico2) / 2
        altura = neckline - cabeca[1]
        return True, {'tipo': 'OCO Invertido', 'altura': altura, 'cabeca': cabeca[1], 'ombro_dir': ombro_d[1], 'neckline': neckline}

def detectar_duplo_topo(df):
    picos = encontrar_picos(df['High'], ordem=5, modo='max')[-2:]
    if len(picos) < 2: return False, {}
    t1, t2 = picos[-2], picos[-1]
    if abs(t1[1] - t2[1]) / t1[1] <= TOLERANCIA_NIVEL:
        vale = min(df['Low'].iloc[t1[0]:t2[0]])
        altura = t1[1] - vale
        return True, {'tipo': 'Topo Duplo', 'altura': altura, 'topo': t1[1], 'vale': vale}
    return False, {}

def detectar_duplo_fundo(df):
    vales = encontrar_picos(df['Low'], ordem=5, modo='min')[-2:]
    if len(vales) < 2: return False, {}
    f1, f2 = vales[-2], vales[-1]
    if abs(f1[1] - f2[1]) / f1[1] <= TOLERANCIA_NIVEL:
        pico = max(df['High'].iloc[f1[0]:f2[0]])
        altura = pico - f1[1]
        return True, {'tipo': 'Fundo Duplo', 'altura': altura, 'fundo': f1[1], 'pico': pico}
    return False, {}

def detectar_triplo_topo(df):
    picos = encontrar_picos(df['High'], ordem=5, modo='max')[-3:]
    if len(picos) < 3: return False, {}
    t1, t2, t3 = picos[-3], picos[-2], picos[-1]
    if all(abs(t[1] - t1[1]) / t1[1] <= TOLERANCIA_NIVEL for t in [t1, t2, t3]):
        vale = min(min(df['Low'].iloc[t1[0]:t2[0]]), min(df['Low'].iloc[t2[0]:t3[0]]))
        altura = t1[1] - vale
        return True, {'tipo': 'Topo Triplo', 'altura': altura, 'topo': t1[1]}
    return False, {}

def detectar_triplo_fundo(df):
    vales = encontrar_picos(df['Low'], ordem=5, modo='min')[-3:]
    if len(vales) < 3: return False, {}
    f1, f2, f3 = vales[-3], vales[-2], vales[-1]
    if all(abs(f[1] - f1[1]) / f1[1] <= TOLERANCIA_NIVEL for f in [f1, f2, f3]):
        pico = max(max(df['High'].iloc[f1[0]:f2[0]]), max(df['High'].iloc[f2[0]:f3[0]]))
        altura = pico - f1[1]
        return True, {'tipo': 'Fundo Triplo', 'altura': altura, 'fundo': f1[1]}
    return False, {}

def detectar_bandeira(df, max_dias=20):
    if len(df) < max_dias + 10: return False, {}
    preco_inicio = df['Close'].iloc[-max_dias-10]
    preco_fim_mastro = df['Close'].iloc[-max_dias]
    mastro = abs(preco_fim_mastro - preco_inicio)
    if mastro / preco_inicio < 0.05: return False, {}
    alt_max = df['High'].iloc[-max_dias:].max()
    alt_min = df['Low'].iloc[-max_dias:].min()
    if (alt_max - alt_min) > 0.5 * mastro: return False, {}
    direcao = 'COMPRA' if preco_fim_mastro > preco_inicio else 'VENDA'
    return True, {'tipo': 'Bandeira', 'mastro': mastro, 'direcao': direcao}

def detectar_flamula(df, max_dias=15):
    if len(df) < max_dias + 10: return False, {}
    preco_inicio = df['Close'].iloc[-max_dias-10]
    preco_fim_mastro = df['Close'].iloc[-max_dias]
    mastro = abs(preco_fim_mastro - preco_inicio)
    if mastro / preco_inicio < 0.05: return False, {}
    highs = df['High'].iloc[-max_dias:]
    lows = df['Low'].iloc[-max_dias:]
    picos = encontrar_picos(highs, ordem=2, modo='max')
    vales = encontrar_picos(lows, ordem=2, modo='min')
    if len(picos) >= 2 and len(vales) >= 2:
        topos_desc = all(picos[i][1] > picos[i+1][1] for i in range(len(picos)-1))
        fundos_asc = all(vales[i][1] < vales[i+1][1] for i in range(len(vales)-1))
        if topos_desc and fundos_asc:
            direcao = 'COMPRA' if preco_fim_mastro > preco_inicio else 'VENDA'
            return True, {'tipo': 'Flâmula', 'mastro': mastro, 'direcao': direcao}
    return False, {}

def detectar_cup_handle(df, max_alca_dias=14):
    if len(df) < 60: return False, {}
    highs = df['High']
    topo_inicio = highs.iloc[-60:].max()
    idx_topo = highs.idxmax()
    apos_topo = df.loc[idx_topo:]
    fundo = apos_topo['Low'].min()
    profundidade = (topo_inicio - fundo) / topo_inicio
    if not (0.3 <= profundidade <= CUP_TOPO_CORRECAO_MAX): return False, {}
    if df['Close'].iloc[-1] < topo_inicio * 0.95: return False, {}
    alca = df.iloc[-max_alca_dias:]
    if len(alca) < 5: return False, {}
    amplitude_alca = (alca['High'].max() - alca['Low'].min()) / alca['Low'].min()
    if amplitude_alca > 0.05: return False, {}
    return True, {'tipo': 'Cup&Handle', 'altura': topo_inicio - fundo, 'resistencia': topo_inicio}

def detectar_diamante(df, min_barras=40):
    if len(df) < min_barras: return False, {}
    metade = min_barras // 2
    primeira = df.iloc[-min_barras:-metade]
    picos1 = encontrar_picos(primeira['High'], ordem=3, modo='max')
    vales1 = encontrar_picos(primeira['Low'], ordem=3, modo='min')
    if len(picos1) < 2 or len(vales1) < 2: return False, {}
    megafone = (picos1[-1][1] > picos1[0][1]) and (vales1[-1][1] < vales1[0][1])
    if not megafone: return False, {}
    segunda = df.iloc[-metade:]
    picos2 = encontrar_picos(segunda['High'], ordem=3, modo='max')
    vales2 = encontrar_picos(segunda['Low'], ordem=3, modo='min')
    if len(picos2) < 2 or len(vales2) < 2: return False, {}
    triangulo = (picos2[-1][1] < picos2[0][1]) and (vales2[-1][1] > vales2[0][1])
    if triangulo:
        altura = max(df['High'].iloc[-min_barras:]) - min(df['Low'].iloc[-min_barras:])
        return True, {'tipo': 'Diamante', 'altura': altura}
    return False, {}

def detectar_canal_tendencia(df, min_toques=2):
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_toques:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_toques:]
    if len(highs) >= min_toques and len(lows) >= min_toques:
        if all(lows[i][1] < lows[i+1][1] for i in range(len(lows)-1)):
            altura = highs[-1][1] - lows[-1][1]
            return True, {'tipo': 'Canal de Alta', 'direcao': 'COMPRA', 'altura': altura}
        if all(highs[i][1] > highs[i+1][1] for i in range(len(highs)-1)):
            altura = highs[-1][1] - lows[-1][1]
            return True, {'tipo': 'Canal de Baixa', 'direcao': 'VENDA', 'altura': altura}
    return False, {}

def detectar_pivo_dow(df, direcao='COMPRA'):
    if len(df) < 30: return False, {}
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-3:]
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-3:]
    if len(lows) < 2 or len(highs) < 2: return False, {}
    if direcao == 'COMPRA':
        if lows[-1][1] > lows[-2][1] and df['Close'].iloc[-1] > highs[-2][1]:
            altura = highs[-2][1] - lows[-2][1]
            return True, {'tipo': 'Pivô de Alta', 'altura': altura}
    else:
        if highs[-1][1] < highs[-2][1] and df['Close'].iloc[-1] < lows[-2][1]:
            altura = highs[-2][1] - lows[-2][1]
            return True, {'tipo': 'Pivô de Baixa', 'altura': altura}
    return False, {}

print("✅ Detecção de padrões carregada")

✅ Detecção de padrões carregada


In [13]:
# ================ MÓDULO DE CONFLUÊNCIA UNIFICADO V16 ================
def calcular_score_confluencia_v16(
    df, direcao, entrada, setup_detectado, info_setup,
    volume_atual, media_volume, rsi_atual, rsi_anterior, mm20, mm50, mm200,
    atr
):
    """
    Retorna (score, vetos, detalhes_sinais, detalhes_camadas)
    Se houver veto, score = -1 e detalhes incluem o motivo.
    """
    score = 0
    sinais_det = []
    camadas = []
    vetos = []

    # 1. VETOS OBRIGATÓRIOS (divergência contrária)
    if USAR_DIVERGENCIAS:
        divergencia_contraria = False
        if detectar_divergencia_oscilador(df, 'IFR', direcao):
            divergencia_contraria = True
            vetos.append('Divergência contrária no IFR')
        if detectar_divergencia_oscilador(df, 'MACD', direcao):
            divergencia_contraria = True
            vetos.append('Divergência contrária no MACD')
        if divergencia_contraria:
            return -1, vetos, ['DIV_BLOQUEADA'], []

    # 2. GAP DE EXAUSTÃO (veto)
    if USAR_GAPS:
        tipo_gap = classificar_gap(df, atr, media_volume)
        if tipo_gap == 'Exaustão':
            vetos.append('Gap de Exaustão detectado')
            return -1, vetos, ['GAP_EXAUSTAO'], []

    # 3. PONTUAÇÃO BASE
    if setup_detectado and info_setup.get('tipo'):
        score += 25
        camadas.append('padrao_grafico')

    # Zona de interesse (suporte/resistência/Fibonacci)
    if preco_em_zona_interesse(df, entrada, direcao):
        score += 25
        camadas.append('zona_interesse')

    # Alinhamento de médias
    if direcao == 'COMPRA':
        if mm20 > mm50 > mm200 and entrada > mm20:
            score += 25
            camadas.append('medias_alinhavadas')
    else:
        if mm20 < mm50 < mm200 and entrada < mm20:
            score += 25
            camadas.append('medias_alinhavadas')

    # IFR/MACD confluência
    if USAR_MACD_CONFLUENCIA:
        macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
        if macd is not None:
            linha_macd = macd['MACD_12_26_9'].iloc[-1]
            linha_sinal = macd['MACDs_12_26_9'].iloc[-1]
            hist_atual = macd['MACDh_12_26_9'].iloc[-1]
            hist_ant = macd['MACDh_12_26_9'].iloc[-2] if len(macd)>=2 else 0
            if direcao == 'COMPRA' and linha_macd > linha_sinal and hist_atual > hist_ant:
                score += 25
                camadas.append('macd_confluencia')
            elif direcao == 'VENDA' and linha_macd < linha_sinal and hist_atual < hist_ant:
                score += 25
                camadas.append('macd_confluencia')
    else:
        # usar IFR apenas
        if direcao == 'COMPRA' and IFR_MIN_COMPRA < rsi_atual < IFR_MAX_COMPRA:
            score += 15
            camadas.append('ifr_alinhado')
        elif direcao == 'VENDA' and IFR_MIN_VENDA < rsi_atual < IFR_MAX_VENDA:
            score += 15
            camadas.append('ifr_alinhado')

    # Volume
    if volume_atual >= media_volume * VOLUME_MULT_MEDIO:
        score += 10
        camadas.append('volume_pico')
    elif volume_atual >= media_volume * 1.2:
        score += 5
        camadas.append('volume_acima_media')

    # OBV antecipatório
    if USAR_OBV and obv_antecipacao(df, direcao):
        score += 15
        camadas.append('obv_antecipacao')

    # Bollinger Squeeze
    if USAR_BOLLINGER and calcular_bollinger_squeeze(df):
        score += 20
        camadas.append('bollinger_squeeze')

    # Gap de Fuga
    if USAR_GAPS:
        tipo_gap = classificar_gap(df, atr, media_volume)
        if tipo_gap == 'Fuga':
            score += 20
            camadas.append('gap_fuga')

    # Divergência a favor
    if USAR_DIVERGENCIAS:
        if detectar_divergencia_oscilador(df, 'IFR', direcao):
            score += 30
            camadas.append('divergencia_favor')
        elif detectar_divergencia_oscilador(df, 'MACD', direcao):
            score += 20
            camadas.append('divergencia_macd_favor')

    # Padrão raro (Diamante, OCO)
    if info_setup.get('tipo') in ['Diamante', 'OCO Topo', 'OCO Invertido']:
        score += 15
        camadas.append('padrao_raro')

    # Candle de força extra (já avaliado externamente, mas podemos adicionar se tiver corpo longo)
    # Será feito na análise principal

    # Sinal independente mínimo
    sinais_det = camadas.copy()  # cada camada é um sinal

    return score, vetos, sinais_det, camadas

print("✅ Módulo de confluência v16 carregado")

✅ Módulo de confluência v16 carregado


In [14]:
# ================ CÁLCULO DE STOP, ALVO E PAYOFF ================
def calcular_stop_por_padrao(df, entrada, direcao, setup_nome, info_setup, atr):
    if direcao == 'COMPRA':
        if setup_nome in ['Triângulo Simétrico', 'Triângulo Ascendente']:
            ultimo_fundo = info_setup.get('fundos', [None])[-1] if 'fundos' in info_setup else None
            return ultimo_fundo - atr if ultimo_fundo else entrada - 1.8*atr
        elif setup_nome == 'Retângulo':
            suporte = info_setup.get('suporte', 0)
            return suporte - atr if suporte else entrada - 1.8*atr
        elif setup_nome in ['OCO Invertido', 'Fundo Duplo', 'Fundo Triplo']:
            fundo_padrao = info_setup.get('fundo', info_setup.get('cabeca', 0))
            return fundo_padrao - atr
        elif setup_nome in ['Bandeira', 'Flâmula']:
            return df['Low'].iloc[-min(20, len(df)):].min() - atr
        elif setup_nome == 'Cup&Handle':
            return info_setup.get('resistencia', entrada) * 0.97
        elif setup_nome == 'Canal de Alta':
            return df['Low'].iloc[-min(10, len(df)):].min() - atr
        elif setup_nome == 'Pivô de Alta':
            return df['Low'].iloc[-min(10, len(df)):].min() - atr
        else:
            return entrada - 1.8 * atr
    else:
        if setup_nome in ['Triângulo Descendente', 'Triângulo Simétrico']:
            ultimo_topo = info_setup.get('topos', [None])[-1] if 'topos' in info_setup else None
            return ultimo_topo + atr if ultimo_topo else entrada + 1.8*atr
        elif setup_nome == 'Retângulo':
            resistencia = info_setup.get('resistencia', 0)
            return resistencia + atr if resistencia else entrada + 1.8*atr
        elif setup_nome in ['OCO Topo', 'Topo Duplo', 'Topo Triplo']:
            topo_padrao = info_setup.get('topo', info_setup.get('cabeca', 0))
            return topo_padrao + atr
        elif setup_nome in ['Bandeira', 'Flâmula']:
            return df['High'].iloc[-min(20, len(df)):].max() + atr
        elif setup_nome == 'Canal de Baixa':
            return df['High'].iloc[-min(10, len(df)):].max() + atr
        elif setup_nome == 'Pivô de Baixa':
            return df['High'].iloc[-min(10, len(df)):].max() + atr
        else:
            return entrada + 1.8 * atr

def calcular_alvo_por_padrao(entrada, direcao, setup_nome, info_setup):
    altura = info_setup.get('altura', info_setup.get('mastro', info_setup.get('profundidade', 0)))
    if altura <= 0:
        return None
    if direcao == 'COMPRA':
        return entrada + altura
    else:
        return entrada - altura

def calcular_payoff(entrada, alvo, stop, direcao, custos=0.003):
    if direcao == 'COMPRA':
        risco = entrada - stop
        retorno = alvo - entrada
    else:
        risco = stop - entrada
        retorno = entrada - alvo
    if risco <= 0:
        return 0.0
    payoff = retorno / risco - custos / risco
    return round(max(0, payoff), 2)

print("✅ Stop, alvo e payoff carregados")

✅ Stop, alvo e payoff carregados


In [15]:
# ================ ANÁLISE PRINCIPAL DE TIMEFRAME (V16) ================
def analisar_timeframe_v16(data_dict, nome_tf, tickers_liquidos, tendencia_superior=None):
    oportunidades = []
    status = []
    contagem_setores = {}
    analisados = 0

    for ticker in tickers_liquidos:
        df_raw = data_dict.get(ticker)
        if df_raw is None or df_raw.empty:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Sem dados'})
            continue

        df = df_raw.copy()
        df.index = pd.to_datetime(df.index)
        df.sort_index(inplace=True)
        if len(df) < 50:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Poucos dados'})
            continue

        analisados += 1
        entrada = df['Close'].iloc[-1]
        if entrada < PRECO_MINIMO:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Preço R$ {entrada:.2f} < {PRECO_MINIMO}'})
            continue

        atr = calcular_atr(df)
        if atr == 0:
            atr = entrada * 0.02
        mm20 = df['Close'].rolling(20).mean().iloc[-1]
        mm50 = df['Close'].rolling(50).mean().iloc[-1]
        mm200 = df['Close'].rolling(200).mean().iloc[-1] if len(df)>=200 else mm50
        rsi_series = ta.rsi(df['Close'], length=14)
        rsi_atual = rsi_series.iloc[-1] if rsi_series is not None and not rsi_series.empty else 50
        rsi_anterior = rsi_series.iloc[-2] if rsi_series is not None and len(rsi_series)>=2 else rsi_atual
        media_volume = df['Volume'].rolling(20).mean().iloc[-1]
        volume_atual = df['Volume'].iloc[-1]

        # Detectar padrão e direção
        setup_detectado = False
        info_setup = {}
        direcao = None
        setup_nome = None

        padroes_compra = [
            (detectar_triangulo_ascendente, 'Triângulo Ascendente'),
            (detectar_triangulo_simetrico, 'Triângulo Simétrico'),
            (detectar_retangulo, 'Retângulo'),
            (detectar_duplo_fundo, 'Fundo Duplo'),
            (detectar_triplo_fundo, 'Fundo Triplo'),
            (lambda x: detectar_oco(x, 'fundo'), 'OCO Invertido'),
            (detectar_bandeira, 'Bandeira'),
            (detectar_flamula, 'Flâmula'),
            (detectar_cup_handle, 'Cup&Handle'),
            (detectar_diamante, 'Diamante'),
            (detectar_canal_tendencia, 'Canal de Alta'),
            (detectar_pivo_dow, 'Pivô de Alta')
        ]

        for func, nome in padroes_compra:
            ok, info = func(df)
            if ok:
                if nome == 'Triângulo Simétrico' and df['Close'].pct_change(20).iloc[-1] <= 0:
                    continue
                if nome == 'Diamante' and df['Close'].iloc[-1] <= df['Close'].iloc[-5]:
                    continue
                direcao = 'COMPRA'
                setup_detectado = True
                info_setup = info
                setup_nome = nome
                break

        if not setup_detectado:
            padroes_venda = [
                (detectar_triangulo_descendente, 'Triângulo Descendente'),
                (detectar_triangulo_simetrico, 'Triângulo Simétrico'),
                (detectar_retangulo, 'Retângulo'),
                (detectar_duplo_topo, 'Topo Duplo'),
                (detectar_triplo_topo, 'Topo Triplo'),
                (lambda x: detectar_oco(x, 'topo'), 'OCO Topo'),
                (detectar_bandeira, 'Bandeira'),
                (detectar_flamula, 'Flâmula'),
                (detectar_diamante, 'Diamante'),
                (detectar_canal_tendencia, 'Canal de Baixa'),
                (detectar_pivo_dow, 'Pivô de Baixa')
            ]
            for func, nome in padroes_venda:
                ok, info = func(df)
                if ok:
                    if nome == 'Triângulo Simétrico' and df['Close'].pct_change(20).iloc[-1] >= 0:
                        continue
                    if nome == 'Diamante' and df['Close'].iloc[-1] >= df['Close'].iloc[-5]:
                        continue
                    direcao = 'VENDA'
                    setup_detectado = True
                    info_setup = info
                    setup_nome = nome
                    break

        if not setup_detectado:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Nenhum padrão detectado'})
            continue

        # ---- FILTRO OBRIGATÓRIO DE CANDLE ----
        candle_atual = df.iloc[-1]
        candle_anterior = df.iloc[-2] if len(df) >= 2 else None
        padroes_candle = classificar_candle(candle_atual, candle_anterior, direcao)

        # Para rompimento normal, precisamos de candle de força
        ok_candle, msg_candle = validar_candle_forca_v2(candle_atual, direcao)
        if not ok_candle:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Candle sem força: {msg_candle}'})
            continue

        # Opção de entrada via pullback (se MODO_PULLBACK = True)
        entrada_real = entrada
        if MODO_PULLBACK and setup_detectado:
            linha_rompida = None
            if direcao == 'COMPRA':
                # Resistência rompida: supor a resistência em info_setup.get('resistencia') ou topo do padrão
                linha_rompida = info_setup.get('resistencia') or info_setup.get('topo')
            else:
                linha_rompida = info_setup.get('suporte') or info_setup.get('fundo')
            if linha_rompida:
                houve_pullback, candle_pb = verificar_pullback(df, linha_rompida, direcao, media_volume)
                if houve_pullback:
                    entrada_real = candle_pb['Close']
                    # adicionar bônus no score depois, por ser pullback validado?

        # Volume
        vol_necessario = media_volume * (VOLUME_MULT_ALTO if setup_nome in ['Triângulo Simétrico','Triângulo Ascendente','Triângulo Descendente','OCO Topo','OCO Invertido','Bandeira','Flâmula','Diamante'] else VOLUME_MULT_MEDIO)
        if volume_atual < vol_necessario:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Volume {volume_atual:.0f} < {vol_necessario:.0f}'})
            continue

        # Cálculo de stop
        stop = calcular_stop_por_padrao(df, entrada_real, direcao, setup_nome, info_setup, atr)

        # Alvos
        alvo_padrao = calcular_alvo_por_padrao(entrada_real, direcao, setup_nome, info_setup)
        if alvo_padrao is None:
            risco = abs(entrada_real - stop)
            alvo_padrao = entrada_real + 3*risco if direcao == 'COMPRA' else entrada_real - 3*risco

        # Fibonacci alvo
        fib_vals = calcular_expansao_fibonacci(df, direcao)
        fib_161 = fib_vals['161.8%'] if fib_vals else None
        alvo_final = min(alvo_padrao, fib_161) if fib_161 else alvo_padrao

        payoff = calcular_payoff(entrada_real, alvo_final, stop, direcao)
        if payoff < PAYOFF_MINIMO:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Payoff {payoff:.2f} < {PAYOFF_MINIMO}'})
            continue

        risco_percent = abs(entrada_real - stop) / entrada_real
        if risco_percent < RISCO_PERCENTUAL_MINIMO or risco_percent > RISCO_PERCENTUAL_MAXIMO:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Risco {risco_percent*100:.1f}% fora dos limites'})
            continue

        # Tendência superior conflito
        if tendencia_superior and ticker in tendencia_superior:
            tend_sup = tendencia_superior[ticker]
            if (direcao == 'COMPRA' and tend_sup == 'BAIXA') or (direcao == 'VENDA' and tend_sup == 'ALTA'):
                status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Tendência {tend_sup} no timeframe superior conflita'})
                continue

        # Confluência V16
        score, vetos, sinais_det, camadas = calcular_score_confluencia_v16(
            df, direcao, entrada_real, setup_detectado, info_setup,
            volume_atual, media_volume, rsi_atual, rsi_anterior,
            mm20, mm50, mm200, atr
        )
        if vetos:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Veto: {", ".join(vetos)}'})
            continue
        if score < PONTUACAO_MINIMA_CONFLUENCIA:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Score {score} < {PONTUACAO_MINIMA_CONFLUENCIA}'})
            continue

        setor = ticker.split('.')[0][:4]
        if contagem_setores.get(setor, 0) >= MAX_ATIVOS_POR_SETOR:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Limite de setor {setor}'})
            continue
        contagem_setores[setor] = contagem_setores.get(setor, 0) + 1

        oportunidade = {
            'Ticker': ticker,
            'Timeframe': nome_tf,
            'Setup': setup_nome,
            'Direcao': direcao,
            'Entrada': round(entrada_real, 2),
            'Stop Loss': round(stop, 2),
            'Alvo': round(alvo_final, 2),
            'Alvo_Estendido': fib_vals['261.8%'] if fib_vals else None,
            'Payoff': payoff,
            'Score': score,
            'Confluencia': {'sinais': len(sinais_det), 'detalhes': sinais_det, 'camadas': camadas},
            'Padroes_Candle': padroes_candle,
            'Instrucao': f"[{nome_tf}] {setup_nome} – {direcao} em R$ {entrada_real:.2f}, stop R$ {stop:.2f}, alvo R$ {alvo_final:.2f}. Payoff {payoff:.2f}:1"
        }
        oportunidades.append(oportunidade)
        status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '✅ APROVADO', 'Motivo': f'Score {score}'})

    return oportunidades, status, analisados

print("✅ Análise de timeframe v16 carregada")

✅ Análise de timeframe v16 carregada


In [16]:
# ================ FLUXO PRINCIPAL V16 ================
logger.inicio("Verificação de Conectividade")
if not verificar_conectividade():
    logger.error("Sem conectividade com a internet!")
    raise SystemExit("Sem conexão com a internet")
logger.fim("Verificação de Conectividade")

logger.inicio("Coleta de Tickers")
if MODO_TESTE:
    tickers_b3 = TESTE_TICKERS.copy()
    logger.log(f"⚠️ MODO TESTE: {len(tickers_b3)} tickers")
else:
    tickers_b3 = obter_tickers_b3()
tickers_sa = [t + ".SA" for t in tickers_b3]
logger.fim("Coleta de Tickers", {'total': len(tickers_b3)})

logger.inicio("Download Histórico (Diário)")
data_d = {}
PERIODO_ANOS = 5
if BRAPI_TOKEN:
    data_d = baixar_dados_brapi(tickers_sa, periodo_anos=PERIODO_ANOS, interval='1d')
if len(data_d) < max(5, len(tickers_sa)*0.5):
    logger.log("Fallback yfinance...")
    data_yf = baixar_dados_yfinance_v2(tickers_sa, periodo=f'{PERIODO_ANOS}y')
    if len(data_yf) > len(data_d):
        data_d = data_yf
logger.fim("Download Histórico", {'sucesso': len(data_d), 'tentados': len(tickers_sa)})

if len(data_d) == 0:
    logger.error("Nenhum dado obtido")
    raise SystemExit("Sem dados para análise")

logger.inicio("Filtro de Liquidez")
tickers_liquidos = []
for t in tickers_sa:
    if t in TICKERS_BLOQUEADOS:
        continue
    df = data_d.get(t)
    if df is None or df.empty:
        continue
    vm = df['Volume'].tail(21).mean()
    pc = df['Close'].iloc[-1]
    if pd.notna(vm) and pd.notna(pc) and vm >= VOLUME_MINIMO_ACAO and (vm * pc) >= VOLUME_FINANCEIRO_MINIMO:
        tickers_liquidos.append(t)
if len(tickers_liquidos) < 10:
    tickers_liquidos = [t+".SA" for t in FALLBACK_TICKERS[:20] if t+".SA" in data_d]
logger.fim("Filtro de Liquidez", {'liquidos': len(tickers_liquidos)})

def resample_tf(df, freq, min_dias=4):
    if df is None or df.empty:
        return None
    agg = {'Open':'first','High':'max','Low':'min','Close':'last','Volume':'sum'}
    dfr = df.resample(freq, closed='right', label='right').agg(agg)
    dfr = dfr.dropna()
    return dfr if len(dfr) >= min_dias else None

logger.inicio("Preparação Multi-TF")
data_diario = {t: data_d[t].copy() for t in tickers_liquidos if t in data_d}
data_semanal = {t: resample_tf(data_d[t], 'W-FRI') for t in tickers_liquidos if t in data_d}
data_mensal = {t: resample_tf(data_d[t], 'ME') for t in tickers_liquidos if t in data_d}
data_semanal = {k:v for k,v in data_semanal.items() if v is not None}
data_mensal = {k:v for k,v in data_mensal.items() if v is not None}
logger.fim("Preparação Multi-TF", {'diario':len(data_diario), 'semanal':len(data_semanal), 'mensal':len(data_mensal)})

tendencia_superior = {}
if ANALISAR_SEMANAL and data_semanal:
    for t, dfw in data_semanal.items():
        if len(dfw) >= 30:
            mm20w = dfw['Close'].rolling(20).mean().iloc[-1]
            mm50w = dfw['Close'].rolling(50).mean().iloc[-1]
            if mm20w > mm50w:
                tendencia_superior[t] = 'ALTA'
            elif mm20w < mm50w:
                tendencia_superior[t] = 'BAIXA'
            else:
                tendencia_superior[t] = 'LATERAL'

todas_oportunidades = []
todos_status = []

if ANALISAR_DIARIO:
    logger.inicio("Análise Diária")
    ops_d, stat_d, _ = analisar_timeframe_v16(data_diario, "Diário", tickers_liquidos, tendencia_superior)
    todas_oportunidades.extend(ops_d)
    todos_status.extend(stat_d)
    logger.fim("Análise Diária", {'aprovados': len(ops_d)})

if ANALISAR_SEMANAL:
    logger.inicio("Análise Semanal")
    ops_s, stat_s, _ = analisar_timeframe_v16(data_semanal, "Semanal", tickers_liquidos, None)
    todas_oportunidades.extend(ops_s)
    todos_status.extend(stat_s)
    logger.fim("Análise Semanal", {'aprovados': len(ops_s)})

if ANALISAR_MENSAL:
    logger.inicio("Análise Mensal")
    ops_m, stat_m, _ = analisar_timeframe_v16(data_mensal, "Mensal", tickers_liquidos, None)
    todas_oportunidades.extend(ops_m)
    todos_status.extend(stat_m)
    logger.fim("Análise Mensal", {'aprovados': len(ops_m)})

todas_oportunidades = sorted(todas_oportunidades, key=lambda x: x['Score'], reverse=True)[:MAX_SETUPS_POR_DIA]
logger.log(f"🎯 Total oportunidades finais: {len(todas_oportunidades)}", "RESULTADO")

[05:22:53] [ETAPA] 🚀 INÍCIO: Verificação de Conectividade
[05:22:53] [ETAPA] ✅ FIM: Verificação de Conectividade (0.0s)
[05:22:53] [ETAPA] 🚀 INÍCIO: Coleta de Tickers
[05:22:54] [ETAPA] ✅ FIM: Coleta de Tickers (0.6s) | total:391
[05:22:54] [ETAPA] 🚀 INÍCIO: Download Histórico (Diário)
[05:22:54] [INFO] Fallback yfinance...
[05:22:54] [INFO] 🔄 yfinance: 1/391
[05:23:08] [INFO] 🔄 yfinance: 21/391
[05:23:19] [INFO] 🔄 yfinance: 41/391
[05:23:29] [INFO] 🔄 yfinance: 61/391
[05:23:40] [INFO] 🔄 yfinance: 81/391
[05:23:52] [INFO] 🔄 yfinance: 101/391
[05:24:03] [INFO] 🔄 yfinance: 121/391
[05:24:15] [INFO] 🔄 yfinance: 141/391
[05:24:26] [INFO] 🔄 yfinance: 161/391
[05:24:37] [INFO] 🔄 yfinance: 181/391
[05:24:47] [INFO] 🔄 yfinance: 201/391
[05:24:58] [INFO] 🔄 yfinance: 221/391
[05:25:09] [INFO] 🔄 yfinance: 241/391


ERROR:yfinance:$AZEV11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")


[05:25:20] [INFO] 🔄 yfinance: 261/391
[05:25:31] [INFO] 🔄 yfinance: 281/391
[05:25:43] [INFO] 🔄 yfinance: 301/391
[05:25:54] [INFO] 🔄 yfinance: 321/391
[05:26:04] [INFO] 🔄 yfinance: 341/391


ERROR:yfinance:$BIOM11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")


[05:26:15] [INFO] 🔄 yfinance: 361/391
[05:26:25] [INFO] 🔄 yfinance: 381/391
[05:26:31] [INFO] 📦 yfinance: 385 tickers
[05:26:31] [ETAPA] 🚀 INÍCIO: Filtro de Liquidez
[05:26:31] [ETAPA] ✅ FIM: Filtro de Liquidez (0.2s) | liquidos:137
[05:26:31] [ETAPA] 🚀 INÍCIO: Preparação Multi-TF
[05:26:33] [ETAPA] ✅ FIM: Preparação Multi-TF (2.5s) | diario:136 | semanal:136 | mensal:136
[05:26:33] [ETAPA] 🚀 INÍCIO: Análise Diária
[05:28:00] [ETAPA] ✅ FIM: Análise Diária (87.1s) | aprovados:0
[05:28:00] [ETAPA] 🚀 INÍCIO: Análise Semanal
[05:28:21] [ETAPA] ✅ FIM: Análise Semanal (20.4s) | aprovados:0
[05:28:21] [ETAPA] 🚀 INÍCIO: Análise Mensal
[05:28:27] [ETAPA] ✅ FIM: Análise Mensal (6.6s) | aprovados:0
[05:28:27] [RESULTADO] 🎯 Total oportunidades finais: 0


In [17]:
# ==================== RELATÓRIO DETALHADO ====================
def calcular_nh_nl(data_dict):
    novas_max, novas_min, total = 0, 0, 0
    for ticker, df in data_dict.items():
        if df is None or len(df) < 52:
            continue
        try:
            max_52 = df['High'].rolling(52).max().iloc[-1]
            min_52 = df['Low'].rolling(52).min().iloc[-1]
            close = df['Close'].iloc[-1]
            if pd.notna(max_52) and pd.notna(min_52):
                total += 1
                if close >= max_52 * 0.995:
                    novas_max += 1
                if close <= min_52 * 1.005:
                    novas_min += 1
        except:
            pass
    saldo = novas_max - novas_min
    pct_max = round(novas_max/total*100,1) if total>0 else 0
    pct_min = round(novas_min/total*100,1) if total>0 else 0
    if saldo > 20:
        diag = "🟢 FORTE (Tendência de alta consistente)"
    elif saldo > 0:
        diag = "🟡 NEUTRO/POSITIVO (Alta moderada)"
    elif saldo > -20:
        diag = "🟠 NEUTRO/NEGATIVO (Baixa moderada)"
    else:
        diag = "🔴 FRACO (Tendência de baixa acentuada)"
    return {'saldo': saldo, 'max': novas_max, 'min': novas_min, 'total': total, 'pct_max': pct_max, 'pct_min': pct_min, 'diagnostico': diag}

def calcular_regime_volatilidade():
    try:
        ibov = yf.download("^BVSP", period="3mo", interval="1d", progress=False)
        if ibov.empty:
            return "N/D"
        ret = ibov['Close'].pct_change().dropna()
        vol_atual = ret.tail(20).std()
        vol_hist = ret.rolling(50).std().mean()
        if vol_atual > vol_hist * 1.2:
            return "ALTA 🚀"
        elif vol_atual < vol_hist * 0.8:
            return "BAIXA 🐢"
        else:
            return "NORMAL ⚖️"
    except:
        return "N/D"

def percentual_acima_media_200(data_dict):
    acima, total = 0, 0
    for ticker, df in data_dict.items():
        if df is None or len(df) < 200:
            continue
        mm200 = df['Close'].rolling(200).mean().iloc[-1]
        if pd.notna(mm200) and df['Close'].iloc[-1] > mm200:
            acima += 1
        total += 1
    return round(acima/total*100,1) if total>0 else 0

def gerar_relatorio_detalhado(ops, todos_status, data_dict, nome_tf="Diário"):
    nhnl = calcular_nh_nl(data_dict)
    regime = calcular_regime_volatilidade()
    pct_acima_200 = percentual_acima_media_200(data_dict)
    recusados = [s for s in todos_status if 'Recusado' in s.get('Status', '')]
    motivos = Counter([s.get('Motivo', 'Desconhecido') for s in recusados])
    top_motivos = motivos.most_common(5)
    watchlist = []
    for s in recusados:
        motivo = s.get('Motivo', '')
        if 'Payoff' not in motivo and 'Risco' not in motivo and 'Candle sem força' not in motivo:
            watchlist.append(f"{s['Ticker']} ({s.get('Timeframe', '?')}) – {motivo[:60]}")
    watchlist = list(dict.fromkeys(watchlist))[:10]
    linhas = []
    linhas.append("=" * 80)
    linhas.append(f"📊 RELATÓRIO GEBRA v16.0 – {datetime.now().strftime('%d/%m/%Y %H:%M')}")
    linhas.append(f"   Timeframe principal: {nome_tf}")
    linhas.append("=" * 80)
    linhas.append("")
    linhas.append("## 🧭 1. CONDIÇÃO DE MERCADO")
    linhas.append(f"- **Regime de volatilidade (Ibovespa)**: {regime}")
    linhas.append(f"- **Ativos acima da MME200**: {pct_acima_200}%")
    linhas.append(f"- **NH‑NL (52 semanas)**: saldo {nhnl['saldo']} (Máx: {nhnl['max']} / Mín: {nhnl['min']} / Total: {nhnl['total']})")
    linhas.append(f"- **Diagnóstico NH‑NL**: {nhnl['diagnostico']}")
    if pct_acima_200 > 60:
        linhas.append("  ➜ Mercado em tendência de ALTA (maioria dos ativos acima da MME200)")
    elif pct_acima_200 < 40:
        linhas.append("  ➜ Mercado em tendência de BAIXA (maioria abaixo da MME200)")
    else:
        linhas.append("  ➜ Mercado lateral / indefinido")
    linhas.append("")
    linhas.append("## ✅ 2. OPORTUNIDADES CONFIRMADAS")
    if ops:
        for i, op in enumerate(ops[:5], 1):
            linhas.append(f"{i}. [{op['Timeframe']}] {op['Ticker']} | {op['Setup']} | {op['Direcao']}")
            linhas.append(f"   Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f} | Alvo: R$ {op['Alvo']:.2f}")
            linhas.append(f"   Payoff: {op['Payoff']:.2f} | Score: {op['Score']} | Sinais: {len(op['Confluencia']['detalhes'])}")
            linhas.append(f"   Camadas: {', '.join(op['Confluencia']['camadas'])}")
            linhas.append(f"   Candles: {', '.join(op.get('Padroes_Candle',[]))}")
            linhas.append("")
    else:
        linhas.append("❌ Nenhuma oportunidade com confluência total foi encontrada.")
        linhas.append("")
    linhas.append("## 🔍 3. PRINCIPAIS MOTIVOS DE RECUSA")
    if top_motivos:
        for motivo, qtd in top_motivos:
            linhas.append(f"   • {motivo}: {qtd} ocorrências")
    else:
        linhas.append("   Nenhum ativo recusado (todos aprovados ou sem dados).")
    linhas.append("")
    linhas.append("## 👀 4. WATCHLIST (QUASE APROVADOS)")
    if watchlist:
        for item in watchlist:
            linhas.append(f"   • {item}")
    else:
        linhas.append("   Nenhum ativo próximo de aprovação.")
    linhas.append("")
    linhas.append("## 💡 5. RECOMENDAÇÃO DO ARQUITETO")
    if nhnl['saldo'] > 10 and pct_acima_200 > 55 and regime != "ALTA 🚀":
        linhas.append("   ✅ Ambiente favorável para operações compradas.")
    elif nhnl['saldo'] < -10 and pct_acima_200 < 45:
        linhas.append("   ⚠️ Tendência de baixa. Priorize operações vendidas.")
    elif regime == "ALTA 🚀":
        linhas.append("   ⚠️ Volatilidade alta! Reduza tamanho e use stops mais largos.")
    else:
        linhas.append("   🟡 Mercado sem direção clara. Aguarde confirmação de tendência.")
    linhas.append("")
    linhas.append("---")
    linhas.append(f"📅 Gerado em {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
    linhas.append("🔧 Para mais detalhes, consulte o arquivo log_gebra_v16.json")
    return "\n".join(linhas)

relatorio = gerar_relatorio_detalhado(todas_oportunidades, todos_status, data_diario, nome_tf="Diário")
with open('relatorio_gebra_v16.txt', 'w', encoding='utf-8') as f:
    f.write(relatorio)

# Envio de e-mail e Telegram (já existentes)
def enviar_email(assunto, corpo):
    if not EMAIL_REMETENTE or not SENHA_APP:
        logger.log("E-mail não configurado – relatório salvo localmente")
        return
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_REMETENTE
        msg['To'] = EMAIL_REMETENTE
        msg['Subject'] = assunto
        msg.attach(MIMEText(corpo, 'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as srv:
            srv.login(EMAIL_REMETENTE, SENHA_APP)
            srv.send_message(msg)
        logger.log("📧 E-mail enviado com relatório detalhado")
    except Exception as e:
        _log_exc('Email', e)

enviar_email(f"📈 GEBRA v16.0 - {len(todas_oportunidades)} ops - {datetime.now().strftime('%d/%m %H:%M')}", relatorio)

def mensagem_telegram(ops):
    if not ops:
        return "📊 GEBRA v16.0\nNenhuma oportunidade. Verifique o relatório completo no e-mail."
    msg = f"🚀 GEBRA v16.0 - {datetime.now().strftime('%d/%m %H:%M')}\n\n"
    for i, op in enumerate(ops[:5], 1):
        msg += f"{i}. <b>[{op['Timeframe']}] {op['Ticker']}</b> | {op['Setup']} | {op['Direcao']}\n"
        msg += f"   Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f}\n"
        msg += f"   Alvo: R$ {op['Alvo']:.2f} | Payoff: {op['Payoff']:.2f}\n\n"
    return msg

enviar_telegram(mensagem_telegram(todas_oportunidades))

# Log JSON
try:
    aprovados = [s for s in todos_status if 'APROVADO' in s.get('Status','')]
    recusados = [s for s in todos_status if 'Recusado' in s.get('Status','')]
    motivos_recusa = Counter([s.get('Motivo','') for s in recusados])
    log_json = {
        'timestamp': datetime.now().isoformat(),
        'status_final': 'concluido',
        'metricas': {
            'total_analisados': len(todos_status),
            'aprovados': len(aprovados),
            'recusados': len(recusados),
            'oportunidades_finais': len(todas_oportunidades)
        },
        'nhnl': calcular_nh_nl(data_diario),
        'regime_volatilidade': calcular_regime_volatilidade(),
        'percentual_acima_mm200': percentual_acima_media_200(data_diario),
        'resumo_recusas': dict(motivos_recusa.most_common(10)),
        'watchlist': [{'ticker': s['Ticker'], 'timeframe': s.get('Timeframe',''), 'motivo': s.get('Motivo','')} for s in recusados if 'Payoff' not in s.get('Motivo','')][:15],
        'oportunidades': todas_oportunidades
    }
    with open('log_gebra_v16.json', 'w', encoding='utf-8') as f:
        json.dump(log_json, f, indent=2, default=str)
    logger.log("📄 Log JSON detalhado salvo")
except Exception as e:
    _log_exc('log_json', e)

gc.collect()
logger.resumo()
print("\n✅ Análise concluída. Relatório: relatorio_gebra_v16.txt")
print(f"   Oportunidades: {len(todas_oportunidades)}")

[05:28:28] [INFO] E-mail não configurado – relatório salvo localmente
[05:28:28] [INFO] 📄 Log JSON detalhado salvo
[05:28:28] [RESUMO] 
[05:28:28] [RESUMO] ⏱️ Tempo total: 335.0s
[05:28:28] [RESUMO]    • Verificação de Conectividade: 0.0s (0%)
[05:28:28] [RESUMO]    • Coleta de Tickers: 0.6s (0%)
[05:28:28] [RESUMO]    • Filtro de Liquidez: 0.2s (0%)
[05:28:28] [RESUMO]    • Preparação Multi-TF: 2.5s (1%)
[05:28:28] [RESUMO]    • Análise Diária: 87.1s (26%)
[05:28:28] [RESUMO]    • Análise Semanal: 20.4s (6%)
[05:28:28] [RESUMO]    • Análise Mensal: 6.6s (2%)
[05:28:28] [RESUMO] ============================================================


✅ Análise concluída. Relatório: relatorio_gebra_v16.txt
   Oportunidades: 0
